# 🚀 Getting Started with the OSPool

Welcome! This notebook walks through the basics of running work on the **Open Science Pool (OSPool)** — computing capacity contributed by institutions across the US and available at no cost to US-affiliated researchers. The OSPool is operated by the [OSG Consortium](https://osg-htc.org) through the [PATh project](https://path-cc.io), and it uses [HTCondor](https://htcondor.org) to manage jobs.

The typical workflow looks like this:

1. Log in to an OSPool **Access Point (AP)** — for example `ap40.uw.osg-htc.org`.
2. Describe your work in an HTCondor **submit file**.
3. Submit it with `condor_submit`, watch it with `condor_q`, and collect the results.

**About this environment.** This notebook comes with a small HTCondor pool (a "minicondor") running inside your Jupyter session. It behaves like an OSPool Access Point — a *pseudo-AP* — so you can practice every command below. On the real OSPool you would type the same commands in a terminal after `ssh`-ing to your AP. Where the OSPool differs from this practice pool, look for the **OSPool note** callouts.

Each grey cell below is a shell command. Click a cell and press **Shift+Enter** to run it.

## 0. Check that the pool is running

Everything in this notebook talks to an HTCondor pool. Let's make sure ours is up (and start it if it isn't).

In [1]:
# Ask the pool for a summary of its resources. If it doesn't answer, start the local pool and give it a few seconds.
condor_status -total 2>/dev/null || { echo "Pool not responding - starting the local HTCondor pool..."; condor_master && sleep 10 && condor_status -total; }


               Total Owner Claimed Unclaimed Matched Preempting  Drain Backfill BkIdle

  X86_64/LINUX     1     0       0         1       0          0      0        0      0

         Total     1     0       0         1       0          0      0        0      0


## 1. Look around the Access Point

A few commands tell you what you're working with.

In [2]:
# Which version of HTCondor is installed?
condor_version

$CondorVersion: 24.3.0 2025-01-03 BuildID: 778135 PackageID: 24.3.0-1+ubu24 GitSHA: a2290360 $
$CondorPlatform: X86_64-Ubuntu_24.04 $


In [3]:
# What machines/slots are available to run jobs? On the OSPool this lists many thousands of slots;
# our practice pool has just one (partitionable) slot on this machine.
condor_status

Name               OpSys      Arch   State     Activity LoadAv Mem    ActvtyTim

slot1@ef9d3d9fe5bb LINUX      X86_64 Unclaimed Idle      0.000 15972  0+00:01:5

               Total Owner Claimed Unclaimed Matched Preempting  Drain Backfill BkIdle

  X86_64/LINUX     1     0       0         1       0          0      0        0      0

         Total     1     0       0         1       0          0      0        0      0


In [4]:
# What's in *your* job queue? (Empty for now.)
condor_q



-- Schedd: jovyan@ef9d3d9fe5bb : <172.17.0.2:9618?... @ 08/17/26 00:50:34
OWNER BATCH_NAME      SUBMITTED   DONE   RUN    IDLE   HOLD  TOTAL JOB_IDS

Total for query: 0 jobs; 0 completed, 0 removed, 0 idle, 0 running, 0 held, 0 suspended 
Total for all users: 0 jobs; 0 completed, 0 removed, 0 idle, 0 running, 0 held, 0 suspended



## 2. Write a small job

A job needs two things: **something to run** and a **submit file** that describes it to HTCondor.

First, a tiny script that reports where and when it ran, and echoes the argument it was given.

In [ ]:
cat << 'EOF' > hello.sh
#!/bin/bash
echo "Hello from the OSPool!"
echo "This job ran on host:  $(hostname)"
echo "It started at:         $(date)"
echo "It received argument:  $1"
sleep 20
echo "Done."
EOF
chmod +x hello.sh
cat hello.sh

Now the submit file. These lines are the core of nearly every OSPool submit file:

* `executable` / `arguments` — what to run.
* `output`, `error`, `log` — where HTCondor puts the job's stdout, stderr, and its event log.
* `request_cpus`, `request_memory`, `request_disk` — what each job needs. HTCondor matches jobs to slots based on these, so keep them honest.
* `queue` — how many copies of the job to submit. Each copy gets its own `$(Process)` number.

In [ ]:
mkdir -p logs

cat << 'EOF' > hello.sub
# What to run
executable = hello.sh
arguments  = $(Process)

# Where HTCondor writes the job's stdout, stderr, and event log
output = logs/hello.$(Cluster).$(Process).out
error  = logs/hello.$(Cluster).$(Process).err
log    = logs/hello.$(Cluster).log

# Resources each job needs
request_cpus   = 1
request_memory = 1GB
request_disk   = 1GB

# How many jobs to submit
queue 1
EOF
cat hello.sub

> **OSPool note.** Two things you'll almost always add on the real OSPool:
> * `transfer_input_files = file1, dir2/` — the AP and the machines that run your jobs do **not** share a filesystem, so list every file your job needs. (Our practice pool happens to run on one machine, so this script works without it.)
> * `+ProjectName = "YourProject"` — if you belong to more than one OSPool project, say which one this work is for.
>
> See the [Roadmap to HTC Workload Submission](https://portal.osg-htc.org/documentation/htc_workloads/workload_planning/roadmap/) for the full picture.

## 3. Submit the job

`condor_submit` reads the submit file, places the job(s) in your queue, and prints the **cluster ID** — the number you'll use to refer to this submission.

In [ ]:
condor_submit hello.sub

## 4. Watch it run

`condor_q` shows the state of your jobs: **idle** (waiting to be matched to a slot), **running**, or **held** (something went wrong — `condor_q -hold` says what).

Re-run the cell below a few times. Our script sleeps for 20 seconds, so you should catch it running before it finishes and disappears from the queue.

In [ ]:
condor_q

In [ ]:
# One row per job instead of the batch summary
condor_q -nobatch

## 5. Look at the results

When a job finishes it leaves the queue. Its stdout landed in the `output` file we named, and `condor_history` remembers completed jobs.

In [ ]:
ls -l logs/

In [ ]:
cat logs/hello.*.out

In [ ]:
# Completed jobs move from the queue to the history
condor_history -limit 5

The `log` file is HTCondor's own record of the job's life — submitted, matched, executing, terminated, plus resource usage. It's the first place to look when something doesn't behave.

In [ ]:
tail -n 30 logs/hello.*.log

## 6. Scale up: many jobs from one submit file

This is the point of high-throughput computing. Change **one line** — `queue 1` becomes `queue 5` — and HTCondor creates 5 independent jobs, numbered `$(Process)` = 0 … 4. Each gets its own argument and its own output file.

In [ ]:
# Edit the queue line (you could also just open hello.sub in the editor and change it by hand)
sed -i 's/^queue .*/queue 5/' hello.sub
grep -n '^queue' hello.sub

condor_submit hello.sub

In [ ]:
condor_q -nobatch

Once the queue is empty again, check that each job got a different `$(Process)` argument (the file name shows `hello.<cluster>.<process>.out`):

In [ ]:
# One line per output file; the file name tells you which cluster.process wrote it
grep "argument" logs/hello.*.out

## 7. Where to go next

You just did the whole loop: write, submit, monitor, collect, scale. On the OSPool the steps are identical — only the size of the pool changes.

* **Get an OSPool account:** [portal.osg-htc.org/application](https://portal.osg-htc.org/application)
* **OSPool documentation:** [portal.osg-htc.org/documentation](https://portal.osg-htc.org/documentation/) — start with the [Quickstart](https://portal.osg-htc.org/documentation/htc_workloads/submitting_workloads/tutorial-quickstart/) and the [Roadmap to HTC Workload Submission](https://portal.osg-htc.org/documentation/htc_workloads/workload_planning/roadmap/)
* **HTCondor manual:** [htcondor.readthedocs.io](https://htcondor.readthedocs.io/)
* **Get help:** email [support@osg-htc.org](mailto:support@osg-htc.org) or drop by [OSPool office hours](https://portal.osg-htc.org/documentation/support_and_training/support/getting-help-from-RCFs/)

Ready for a real workload? The next recipe processes a whole folder of files, one job per file:

<button data-commandLinker-command="docmanager:open"
        data-commandLinker-args='{"path": "recipe-demos/python-process-folder.ipynb"}' href="#">
  Open the next recipe: Processing a folder of files with a Python script
</button>

## Clean up (optional)

Remove the files this notebook created so you can run it again from a clean slate.

In [ ]:
rm -f hello.sh hello.sub logs/hello.*
condor_q